### Connect to MongoDB

In [1]:
# Lets create connection to MongoDB and select database and collection

import os
from pymongo import MongoClient
import pymongo
from dotenv import load_dotenv


load_dotenv("../.env")

user = os.getenv("MONGO_USER")
password = os.getenv("MONGO_PASSWORD")
host = os.getenv("MONGO_HOST")
port = os.getenv("MONGO_PORT")

uri = f"mongodb://{user}:{password}@{host}:{port}/"

my_client = MongoClient(uri)
my_db = my_client.test
my_col= my_db.testcol

# Clear collection before demo
my_col.delete_many({})

DeleteResult({'n': 6, 'ok': 1.0}, acknowledged=True)

### Write simple document to MongoDB

In [2]:
# Let's create documents to insert. Documents can be anything that can be represented as a JSON (dictionaries, lists, nested structures etc.)

simple_clients=[
{ "Customer_id": "1", "Country": "DE" },
{ "Customer_id": "2", "Country": "US" },
{ "Customer_id": "3", "Country": "PL" }
]

#write documents to collection. We will use insert_one() (there is also insert_many()) method which return an object with attributes 
# related to write (e.g. inserted_ids) useful for debugging and logging.
write=my_col.insert_many(simple_clients)
print(f"{len(write.inserted_ids)} were written and following IDs were generated: {write.inserted_ids}")

#Lets print all documents. We use collection.find() method which returns a cursor (iterable object). One needs to loop through that object to get documents.
for doc in my_col.find():
    print(doc)

3 were written and following IDs were generated: [ObjectId('69bd40f3972a6de492984bfd'), ObjectId('69bd40f3972a6de492984bfe'), ObjectId('69bd40f3972a6de492984bff')]
{'_id': ObjectId('69bd40f3972a6de492984bfd'), 'Customer_id': '1', 'Country': 'DE'}
{'_id': ObjectId('69bd40f3972a6de492984bfe'), 'Customer_id': '2', 'Country': 'US'}
{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL'}


### List databases, collection in a database, documents in collection, and a document meeting filter criteria.

In [3]:
# List all databases
print(my_client.list_database_names(),"\n")
# List all collections in the our database
print(my_db.list_collection_names(),"\n")
# List all documents in our collection. collection.find() returns a cursor (iterable object), so we need to loop through it to get documents.
for doc in my_col.find({}):
    print(doc)
print("\n")
#List document meeting filter criteria. Filter is a dictionary with field name as key and value to filter as value. 
filter = {"Customer_id": "1"}
for doc in my_col.find(filter):
    print(doc)


['admin', 'config', 'local', 'test'] 

['testcol'] 

{'_id': ObjectId('69bd40f3972a6de492984bfd'), 'Customer_id': '1', 'Country': 'DE'}
{'_id': ObjectId('69bd40f3972a6de492984bfe'), 'Customer_id': '2', 'Country': 'US'}
{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL'}


{'_id': ObjectId('69bd40f3972a6de492984bfd'), 'Customer_id': '1', 'Country': 'DE'}


### Update simple document

In [4]:
# Lets update a document by adding new attribute, changing existing one and deleting and existing one.
# Will be using collection.update_one() method which takes filter to select document to update and new value.
# New values are passed as a dictionary with operator as key (e.g. $set for adding/updating attribute, $unset for deleting attribute) 
 
#print document before update
filter = {"Customer_id": "1"}
for doc in my_col.find(filter):
    print(doc)
print("\n")

#add new attribute "Name" with value "Frank" to document with Customer_id 1
filter = {"Customer_id": "1"}
newvalue = {"$set": {"Name": "Frank"}}
my_col.update_one(filter, newvalue)

#print document after adding new attribute
for doc in my_col.find(filter):
    print(doc)
print("\n")

#lets update existing attribute "Country" to "USA" to document with Customer_id 1
filter = {"Customer_id": "1"}
newvalues = {"$set": {"Country":"USA"}}
my_col.update_one(filter, newvalues)

#print document after updating existing attribute
for doc in my_col.find(filter):
    print(doc)
print("\n")
#lets delete existing attribute "Country" from document with Customer_id 1
filter = {"Customer_id": "1"}
newvalues = {"$unset": {"Name": ""}}
my_col.update_one(filter, newvalues)

#print document after deleting existing attribute
for doc in my_col.find(filter):
    print(doc)


{'_id': ObjectId('69bd40f3972a6de492984bfd'), 'Customer_id': '1', 'Country': 'DE'}


{'_id': ObjectId('69bd40f3972a6de492984bfd'), 'Customer_id': '1', 'Country': 'DE', 'Name': 'Frank'}


{'_id': ObjectId('69bd40f3972a6de492984bfd'), 'Customer_id': '1', 'Country': 'USA', 'Name': 'Frank'}


{'_id': ObjectId('69bd40f3972a6de492984bfd'), 'Customer_id': '1', 'Country': 'USA'}


### Write documents containing nested document

In [5]:
# We will insert documents with an "Order_items" attribute that contains a nested document with details about the order items for each customer. 

complex_clients=[
{ "Customer_id": "4", "Country": "FR", "Order_items": {"StockCode":"1","UnitPrice":10, "Description":"Fluffy Bear" ,"Quantity" : 10}},
{ "Customer_id": "5", "Country": "ES", "Order_items": {"StockCode":"2","UnitPrice":20, "Description":"Cuddly Lion" ,"Quantity" : 5}},
{ "Customer_id": "6", "Country": "SE", "Order_items": {"StockCode":"3","UnitPrice":15, "Description":"Soft Tiger" ,"Quantity" : 8}}
]

In [6]:
# Let's write these documents to the collection and print the inserted ids
write=my_col.insert_many(complex_clients)
print(f"{len(write.inserted_ids)} were written and following IDs were generated: {write.inserted_ids}") 

3 were written and following IDs were generated: [ObjectId('69bd4104972a6de492984c00'), ObjectId('69bd4104972a6de492984c01'), ObjectId('69bd4104972a6de492984c02')]


In [7]:
# List all documents in the collection to see the effect of the insert operation
print("All documents in the collection:")
for doc in my_col.find():
    print(doc)

# Print documents meeting filter criteria on nested attribute (e.g. StockCode in order_items)
filter = {"Order_items.StockCode": "1"}
print("Documents with StockCode '1':")
for doc in my_col.find(filter):
    print(doc)

All documents in the collection:
{'_id': ObjectId('69bd40f3972a6de492984bfd'), 'Customer_id': '1', 'Country': 'USA'}
{'_id': ObjectId('69bd40f3972a6de492984bfe'), 'Customer_id': '2', 'Country': 'US'}
{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL'}
{'_id': ObjectId('69bd4104972a6de492984c00'), 'Customer_id': '4', 'Country': 'FR', 'Order_items': {'StockCode': '1', 'UnitPrice': 10, 'Description': 'Fluffy Bear', 'Quantity': 10}}
{'_id': ObjectId('69bd4104972a6de492984c01'), 'Customer_id': '5', 'Country': 'ES', 'Order_items': {'StockCode': '2', 'UnitPrice': 20, 'Description': 'Cuddly Lion', 'Quantity': 5}}
{'_id': ObjectId('69bd4104972a6de492984c02'), 'Customer_id': '6', 'Country': 'SE', 'Order_items': {'StockCode': '3', 'UnitPrice': 15, 'Description': 'Soft Tiger', 'Quantity': 8}}
Documents with StockCode '1':
{'_id': ObjectId('69bd4104972a6de492984c00'), 'Customer_id': '4', 'Country': 'FR', 'Order_items': {'StockCode': '1', 'UnitPrice': 10, 'Description'

### Update documents containing nested document

In [8]:
# Let's update an attribute of the nested document. We will change the description of the order item with StockCode "2" and print the updated document

filter = {"Order_items.StockCode": "1"}

print('Document before update:')
for doc in my_col.find(filter):
    print(doc)

# To update a nested document, we need to use the dot notation to specify the path to the attribute we want to update. 
newvalue={'$set':{'Order_items.Description':'Ultra soft squishy monkey'}}
my_col.update_one(filter,newvalue)

print('Document after update:')
for doc in my_col.find(filter):
    print(doc)

Document before update:
{'_id': ObjectId('69bd4104972a6de492984c00'), 'Customer_id': '4', 'Country': 'FR', 'Order_items': {'StockCode': '1', 'UnitPrice': 10, 'Description': 'Fluffy Bear', 'Quantity': 10}}
Document after update:
{'_id': ObjectId('69bd4104972a6de492984c00'), 'Customer_id': '4', 'Country': 'FR', 'Order_items': {'StockCode': '1', 'UnitPrice': 10, 'Description': 'Ultra soft squishy monkey', 'Quantity': 10}}


### Write documents containing an array of documents

In [9]:
# lets write and read an array of nested documents to demonstrate how to work with arrays in MongoDB
array_order_items = [
    {"StockCode":"6","UnitPrice":5, "Description":"Soft Elephant" ,"Quantity" : 12},
    {"StockCode":"7","UnitPrice":8, "Description":"Fluffy Giraffe" ,"Quantity" : 7}
]

filter = {"Customer_id": "3"}

print('Document before update:')
for doc in my_col.find(filter):
    print(doc)

newvalue={'$set':{'Order_items':array_order_items}}
my_col.update_one(filter,newvalue)

print('Document after update:')
for doc in my_col.find(filter):
    print(doc)

Document before update:
{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL'}
Document after update:
{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL', 'Order_items': [{'StockCode': '6', 'UnitPrice': 5, 'Description': 'Soft Elephant', 'Quantity': 12}, {'StockCode': '7', 'UnitPrice': 8, 'Description': 'Fluffy Giraffe', 'Quantity': 7}]}


In [10]:
filter = {"Customer_id": "3"}
new_order_item = {"StockCode":"8","UnitPrice":12, "Description":"Cuddly Zebra" ,"Quantity" : 3}

# Let's print the document before we add a new item to the array of order items
print('Document before update:')
for doc in my_col.find(filter):
    print(doc)

# To add a new item to the array of order items, we can use the $push operator
newvalue={'$push':{'Order_items':new_order_item}}

my_col.update_one(filter,newvalue)

# Let's print the document after we add a new item to the array of order items
print('Document after update:')
for doc in my_col.find(filter):
    print(doc)

# Let's update the description of the order item on client 3 with StockCode "6" and print the updated document
query = {"Customer_id": "3", "Order_items.StockCode": "8"}
newvalue={'$set':{'Order_items.$.Description': 'Updated Description'}}
my_col.update_one(query,newvalue)

# Let's print the document before we update the quantity of the order item with StockCode "8"
print('Document after deletion:')
for doc in my_col.find(filter):
    print(doc)


# To delete an item from the array of order items, we can use the $pull operator. Let's delete the order item with StockCode "8" and print the updated document
filter = {"Customer_id": "3"}
cancel_order_item = {"$pull": {"Order_items": {"StockCode": "8"}}}
my_col.update_one(filter, cancel_order_item)

print('Document after deletion:')
for doc in my_col.find(filter):
    print(doc)

Document before update:
{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL', 'Order_items': [{'StockCode': '6', 'UnitPrice': 5, 'Description': 'Soft Elephant', 'Quantity': 12}, {'StockCode': '7', 'UnitPrice': 8, 'Description': 'Fluffy Giraffe', 'Quantity': 7}]}
Document after update:
{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL', 'Order_items': [{'StockCode': '6', 'UnitPrice': 5, 'Description': 'Soft Elephant', 'Quantity': 12}, {'StockCode': '7', 'UnitPrice': 8, 'Description': 'Fluffy Giraffe', 'Quantity': 7}, {'StockCode': '8', 'UnitPrice': 12, 'Description': 'Cuddly Zebra', 'Quantity': 3}]}
Document after deletion:
{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL', 'Order_items': [{'StockCode': '6', 'UnitPrice': 5, 'Description': 'Soft Elephant', 'Quantity': 12}, {'StockCode': '7', 'UnitPrice': 8, 'Description': 'Fluffy Giraffe', 'Quantity': 7}, {'StockCode': '8', 'UnitPrice': 12, 'Descr

In [ ]:
# Lets print documents where quantity of order items is greater than 
# We can use dot notation to filter on nested attributes in arrays.
# experiment with different operators (e.g. $gte greater than or equal, $lt less than, $eq equal, $ne not equal etc.) to filter documents based on nested attributes in arrays.
for doc in my_col.find({"Order_items.Quantity": {"$gt": 3}}):
    print(doc)

{'_id': ObjectId('69bd40f3972a6de492984bff'), 'Customer_id': '3', 'Country': 'PL', 'Order_items': [{'StockCode': '6', 'UnitPrice': 5, 'Description': 'Soft Elephant', 'Quantity': 12}, {'StockCode': '7', 'UnitPrice': 8, 'Description': 'Fluffy Giraffe', 'Quantity': 7}]}
{'_id': ObjectId('69bd4104972a6de492984c00'), 'Customer_id': '4', 'Country': 'FR', 'Order_items': {'StockCode': '1', 'UnitPrice': 10, 'Description': 'Ultra soft squishy monkey', 'Quantity': 10}}
{'_id': ObjectId('69bd4104972a6de492984c01'), 'Customer_id': '5', 'Country': 'ES', 'Order_items': {'StockCode': '2', 'UnitPrice': 20, 'Description': 'Cuddly Lion', 'Quantity': 5}}
{'_id': ObjectId('69bd4104972a6de492984c02'), 'Customer_id': '6', 'Country': 'SE', 'Order_items': {'StockCode': '3', 'UnitPrice': 15, 'Description': 'Soft Tiger', 'Quantity': 8}}


In [25]:
# Lets get total quantity of order items per country. We will use aggregation pipeline for that. 
# Aggregation pipeline is a framework for data aggregation modeled on the concept of data processing pipelines.
# Documents enter a multi-stage pipeline that transforms the documents into aggregated results. 
# Each stage performs an operation on the input documents and passes the result to the next stage. 
# In our case we will use $unwind stage to deconstruct the Order_items array and $group stage to group documents by country 
# and calculate total quantity of order items per country.

pipeline = [
    {"$unwind": "$Order_items"},   # deconstruct the Order_items array to get one document per order item
    {"$group": {                   # group documents by county
        "_id": "$Country",
        "total_quantity": {"$sum": "$Order_items.Quantity"},
        "avg_price": {"$avg": "$Order_items.UnitPrice"},
        "min_price": {"$min": "$Order_items.UnitPrice"}
    }},
    {"$sort": {"total_quantity": -1}} # sort results by total quantity in descending order
]

# return json with total quantity of order items per country
result = list(my_col.aggregate(pipeline))
display(result)

# convert result to pandas dataframe for better visualization and rename _id column to Country
import pandas as pd
df = pd.DataFrame(result)
df = df.rename(columns={"_id": "Country"})

display(df)

[{'_id': 'PL', 'total_quantity': 19, 'avg_price': 6.5, 'min_price': 5},
 {'_id': 'FR', 'total_quantity': 10, 'avg_price': 10.0, 'min_price': 10},
 {'_id': 'SE', 'total_quantity': 8, 'avg_price': 15.0, 'min_price': 15},
 {'_id': 'ES', 'total_quantity': 5, 'avg_price': 20.0, 'min_price': 20}]

,Country,total_quantity,avg_price,min_price
0,PL,19,6.5,5
1,FR,10,10.0,10
2,SE,8,15.0,15
3,ES,5,20.0,20
